# Sprint 5: Policy & Big Tech Layer — Exploration

**Goal:** Map AI companies' biosecurity evaluation relationships, policy partnerships, and government advisory roles.

**Approach:** Dry-run mode — fetch evidence first, present stats, define relationships iteratively with human review.

**Key question:** What *kind* of relationships exist between frontier AI labs and biosecurity orgs? We need to define edge types carefully.

### Candidate Relationships (from ROADMAP)
| Relationship | Source Type | Status |
|---|---|---|
| Anthropic → SecureBio | Blog posts, eval reports | TBD |
| OpenAI → Gryphon Scientific | Press release, papers | TBD |
| OpenAI → LANL | Press release | TBD |
| Anthropic/OpenAI/DeepMind → NTI Bio | Forum participation | TBD |
| SecureBio → NIST | Standards committee | TBD |

### Already in graph
From Sprint 1 (workshop author affiliations): `inst_SecureBio`, `inst_Anthropic`, `inst_OpenAI`, `inst_Google_DeepMind`, `inst_Meta`, `inst_Microsoft`

From Sprint 2/3 (Coefficient grants): `org_gryphon_scientific`, `org_federation_of_american_scientists`, `org_beri_berkeley_existential_risk_initiative`

In [1]:
import json, os, re
from collections import Counter, defaultdict

BASE = os.path.abspath('../../')
RAW_DIR = os.path.join(BASE, 'data', 'raw', 's5_policy_bigtech')
os.makedirs(RAW_DIR, exist_ok=True)

with open(os.path.join(BASE, 'data', 'graph_data.json')) as f:
    graph = json.load(f)

nodes = graph['nodes']
edges = graph['edges']
print(f'Current graph: {len(nodes)} nodes, {len(edges)} edges')

# Index existing nodes
node_by_id = {n['id']: n for n in nodes}
node_by_label_lower = {n['label'].lower(): n['id'] for n in nodes}

Current graph: 553 nodes, 825 edges


## Step 1: Inventory existing AI/biosecurity nodes

Before adding relationships, let's see what we already have.

In [2]:
# AI frontier labs already in graph
AI_LABS = ['anthropic', 'openai', 'google deepmind', 'deepmind', 'meta', 'meta ai',
           'microsoft', 'google', 'google brain']

# Biosecurity policy orgs to look for
BIOSEC_ORGS = ['securebio', 'gryphon', 'nti', 'nuclear threat initiative',
               'federation of american scientists', 'johns hopkins center',
               'center for health security', 'nist', 'beri', 'cset', 'rand',
               'cdc', 'who', 'bwc', 'ibbis', 'igem']

print('=== AI FRONTIER LABS IN GRAPH ===')
ai_nodes = []
for n in nodes:
    for kw in AI_LABS:
        if kw == n['label'].lower() or kw in n['label'].lower():
            ai_nodes.append(n)
            # Count edges
            n_edges = sum(1 for e in edges if e['source'] == n['id'] or e['target'] == n['id'])
            print(f"  [{n['type']}] {n['id']}: {n['label']} ({n_edges} edges)")
            break

print(f'\nTotal AI lab nodes: {len(ai_nodes)}')

print('\n=== BIOSECURITY/POLICY ORGS IN GRAPH ===')
biosec_nodes = []
for n in nodes:
    for kw in BIOSEC_ORGS:
        if kw in n['label'].lower():
            biosec_nodes.append(n)
            n_edges = sum(1 for e in edges if e['source'] == n['id'] or e['target'] == n['id'])
            print(f"  [{n['type']}] {n['id']}: {n['label']} ({n_edges} edges)")
            break

print(f'\nTotal biosec/policy org nodes: {len(biosec_nodes)}')

=== AI FRONTIER LABS IN GRAPH ===
  [presentation] paper_7FGZVr6vlK: Robust LLM Unlearning with MUDMAN: Meta-Unlearning with Disruption Masking And Normalization (4 edges)
  [institution] inst_Meta: Meta (1 edges)
  [institution] inst_Google: Google (3 edges)
  [institution] inst_Google_DeepMind: Google DeepMind (3 edges)
  [institution] inst_Anthropic: Anthropic (3 edges)
  [institution] inst_Microsoft: Microsoft (1 edges)
  [institution] inst_DeepMind: DeepMind (1 edges)
  [institution] inst_OpenAI: OpenAI (1 edges)
  [institution] inst_Google_Brain: Google Brain (1 edges)
  [institution] inst_Meta_AI: Meta AI (1 edges)
  [org] org_jake_pencharz_metagenomic_sequencing_research: Jake Pencharz — Metagenomic Sequencing Research (1 edges)

Total AI lab nodes: 11

=== BIOSECURITY/POLICY ORGS IN GRAPH ===
  [presentation] paper_DOsb0ZbtjA: Behavioral Red Teaming: Investigating Future Biosecurity Risk from Agentic AI and De Novo Sequence Design (1 edges)
  [presentation] paper_fDysOrWaGd: A

## Step 2: Evidence gathering — Anthropic biosecurity evaluations

Search for public evidence of Anthropic's biosecurity evaluation partnerships.

In [3]:
# This cell documents URLs to fetch and what we find.
# Actual fetching done via WebFetch tool (outside notebook), results saved to RAW_DIR.

EVIDENCE_TRACKER = []

def add_evidence(source_url, relationship, actors, evidence_type, confidence, notes=''):
    """Track an evidence item for a potential graph edge."""
    entry = {
        'source_url': source_url,
        'relationship': relationship,
        'actors': actors,
        'evidence_type': evidence_type,  # 'blog_post', 'press_release', 'paper', 'webpage', 'news_article'
        'confidence': confidence,  # 'HIGH', 'MEDIUM', 'LOW'
        'notes': notes,
        'fetched': False,
        'verified': False
    }
    EVIDENCE_TRACKER.append(entry)
    return len(EVIDENCE_TRACKER) - 1

print('Evidence tracker initialized. Will populate as we fetch sources.')

Evidence tracker initialized. Will populate as we fetch sources.


In [4]:
# === ANTHROPIC BIOSECURITY EVIDENCE ===
# Known public sources to check:

add_evidence(
    source_url='https://www.anthropic.com/research/biosecurity-safeguards',
    relationship='biosecurity_evaluation',
    actors=['Anthropic', 'SecureBio'],
    evidence_type='blog_post',
    confidence='HIGH',
    notes='Anthropic blog about biosecurity safeguards and SecureBio evaluation'
)

add_evidence(
    source_url='https://securebio.substack.com/',
    relationship='biosecurity_evaluation',
    actors=['SecureBio', 'Anthropic'],
    evidence_type='blog_post',
    confidence='MEDIUM',
    notes='SecureBio substack - may contain posts about Anthropic eval work'
)

# === OPENAI BIOSECURITY EVIDENCE ===
add_evidence(
    source_url='https://openai.com/index/building-an-early-warning-system-for-llm-aided-biological-threat-creation/',
    relationship='biosecurity_evaluation',
    actors=['OpenAI', 'Gryphon Scientific'],
    evidence_type='blog_post',
    confidence='HIGH',
    notes='OpenAI blog about Gryphon Scientific bio threat eval. May also mention RAND.'
)

add_evidence(
    source_url='https://openai.com/index/gpt-4o-system-card/',
    relationship='biosecurity_evaluation',
    actors=['OpenAI', 'multiple orgs'],
    evidence_type='system_card',
    confidence='MEDIUM',
    notes='GPT-4o system card may list bio eval partners'
)

# === GOOGLE DEEPMIND BIOSECURITY EVIDENCE ===
add_evidence(
    source_url='https://deepmind.google/about/responsibility-safety/',
    relationship='biosecurity_evaluation',
    actors=['Google DeepMind', 'unknown orgs'],
    evidence_type='webpage',
    confidence='LOW',
    notes='DeepMind safety page - may mention bio eval partners'
)

# === NTI BIOSECURITY EVIDENCE ===
add_evidence(
    source_url='https://www.nti.org/about/biosecurity/',
    relationship='policy_participation',
    actors=['NTI', 'multiple AI labs'],
    evidence_type='webpage',
    confidence='MEDIUM',
    notes='NTI biosecurity program - check for AI company partnerships'
)

# === META / LLAMA BIOSECURITY ===
add_evidence(
    source_url='https://ai.meta.com/blog/meta-llama-3-1/',
    relationship='biosecurity_evaluation',
    actors=['Meta', 'unknown orgs'],
    evidence_type='blog_post',
    confidence='LOW',
    notes='Llama 3.1 release - check if bio eval partners mentioned'
)

print(f'Registered {len(EVIDENCE_TRACKER)} evidence items to fetch.')
for i, e in enumerate(EVIDENCE_TRACKER):
    print(f"  [{i}] {e['confidence']:6s} | {' ↔ '.join(e['actors']):40s} | {e['source_url'][:70]}")

Registered 7 evidence items to fetch.
  [0] HIGH   | Anthropic ↔ SecureBio                    | https://www.anthropic.com/research/biosecurity-safeguards
  [1] MEDIUM | SecureBio ↔ Anthropic                    | https://securebio.substack.com/
  [2] HIGH   | OpenAI ↔ Gryphon Scientific              | https://openai.com/index/building-an-early-warning-system-for-llm-aide
  [3] MEDIUM | OpenAI ↔ multiple orgs                   | https://openai.com/index/gpt-4o-system-card/
  [4] LOW    | Google DeepMind ↔ unknown orgs           | https://deepmind.google/about/responsibility-safety/
  [5] MEDIUM | NTI ↔ multiple AI labs                   | https://www.nti.org/about/biosecurity/
  [6] LOW    | Meta ↔ unknown orgs                      | https://ai.meta.com/blog/meta-llama-3-1/


## Step 3: Fetch and analyze sources

Each source will be fetched, saved to `data/raw/s5_policy_bigtech/`, and analyzed for relationship evidence.

**Results are populated below as WebFetch calls return.**

In [5]:
# === RESULTS FROM WEB FETCHING (2026-03-12) ===
# Sources fetched via WebFetch/WebSearch, saved to data/raw/s5_policy_bigtech/

FINDINGS = {
    'securebio_evals': {
        'source': 'securebio.substack.com + epoch.ai analysis',
        'raw_file': 'securebio_eval_overview.md',
        'relationships': [
            {'evaluator': 'SecureBio', 'evaluated': 'Anthropic', 'models': ['Claude 3.7 Sonnet', 'Claude 4'], 'eval_type': 'VCT + CBRN assessment', 'confidence': 'HIGH'},
            {'evaluator': 'SecureBio', 'evaluated': 'OpenAI', 'models': ['GPT-4.5', 'o3-mini', 'o4-mini'], 'eval_type': 'VCT', 'confidence': 'HIGH'},
            {'evaluator': 'SecureBio', 'evaluated': 'Google DeepMind', 'models': ['Gemini 2.5 Pro'], 'eval_type': 'VCT', 'confidence': 'HIGH'},
            {'evaluator': 'SecureBio', 'evaluated': 'xAI', 'models': ['unspecified'], 'eval_type': 'risk management eval', 'confidence': 'MEDIUM'},
        ],
        'also_mentioned': ['Deloitte (co-developed virology tasks)', 'Signature Science (co-developed virology tasks)']
    },
    'openai_gryphon': {
        'source': 'openai.com blog + venturebeat coverage',
        'raw_file': 'rand_bio_threat_studies.md',
        'relationships': [
            {'evaluator': 'Gryphon Scientific', 'evaluated': 'OpenAI', 'models': ['GPT-4'], 'eval_type': 'bio threat creation eval (5 stages)', 'confidence': 'HIGH'},
        ],
        'study_details': '100 participants (50 PhD + 50 students), uplift methodology, mean uplift 0.88/10 (not significant)'
    },
    'rand_studies': {
        'source': 'rand.org publications',
        'raw_file': 'rand_bio_threat_studies.md',
        'relationships': [
            {'evaluator': 'RAND Corporation', 'evaluated': 'multiple LLMs', 'models': ['30+ models'], 'eval_type': 'red team bio attack planning', 'confidence': 'HIGH'},
        ],
        'study_details': 'No significant difference in bio attack plan viability with/without LLMs'
    },
    'nti_aixbio': {
        'source': 'nti.org + web search',
        'raw_file': 'nti_aixbio_program.md',
        'relationships': [
            {'convener': 'NTI', 'participants': ['Google DeepMind', 'OpenAI', 'Anthropic'], 'activity': 'AIxBio Global Forum', 'confidence': 'MEDIUM'},
        ],
        'note': 'NTI role is CONVENING/POLICY, not direct evaluation. No contractual relationship with AI labs.'
    },
    'uk_aisi': {
        'source': 'epoch.ai analysis + fedscoop article',
        'raw_file': 'epoch_biorisk_eval_analysis.md',
        'relationships': [
            {'evaluator': 'UK AI Safety Institute', 'evaluated': 'Anthropic', 'models': ['Claude 3.5 Sonnet'], 'eval_type': 'government safety evaluation', 'confidence': 'HIGH'},
            {'evaluator': 'UK AI Safety Institute', 'evaluated': 'OpenAI', 'models': ['o1'], 'eval_type': 'government safety evaluation', 'confidence': 'HIGH'},
        ]
    },
    'futurehouse': {
        'source': 'epoch.ai analysis',
        'raw_file': 'epoch_biorisk_eval_analysis.md',
        'relationships': [
            {'evaluator': 'FutureHouse', 'evaluated': 'multiple labs', 'models': ['various'], 'eval_type': 'LAB-Bench benchmark', 'confidence': 'MEDIUM'},
        ]
    }
}

# Count total relationships found
total_rels = sum(len(f.get('relationships', [])) for f in FINDINGS.values())
print(f'=== EVIDENCE GATHERING COMPLETE ===')
print(f'Sources analyzed: {len(FINDINGS)}')
print(f'Total relationships found: {total_rels}')
print()
for name, data in FINDINGS.items():
    print(f'  {name}: {len(data["relationships"])} relationships')
    for r in data['relationships']:
        if 'evaluator' in r:
            print(f'    {r["evaluator"]} → {r["evaluated"]} ({r["confidence"]})')
        elif 'convener' in r:
            print(f'    {r["convener"]} convenes {", ".join(r["participants"])} ({r["confidence"]})')

=== EVIDENCE GATHERING COMPLETE ===
Sources analyzed: 6
Total relationships found: 10

  securebio_evals: 4 relationships
    SecureBio → Anthropic (HIGH)
    SecureBio → OpenAI (HIGH)
    SecureBio → Google DeepMind (HIGH)
    SecureBio → xAI (MEDIUM)
  openai_gryphon: 1 relationships
    Gryphon Scientific → OpenAI (HIGH)
  rand_studies: 1 relationships
    RAND Corporation → multiple LLMs (HIGH)
  nti_aixbio: 1 relationships
    NTI convenes Google DeepMind, OpenAI, Anthropic (MEDIUM)
  uk_aisi: 2 relationships
    UK AI Safety Institute → Anthropic (HIGH)
    UK AI Safety Institute → OpenAI (HIGH)
  futurehouse: 1 relationships
    FutureHouse → multiple labs (MEDIUM)


## Step 4: Define relationship types

Before adding edges, we need to define what relationship types make sense for this layer.

**Candidate edge types for Sprint 5:**

| Edge Type | Meaning | Example |
|---|---|---|
| `biosecurity_eval` | Org A evaluated Org B's model for bio risk | SecureBio evaluated Anthropic's Claude |
| `policy_partner` | Orgs collaborate on biosecurity policy | Anthropic + NTI at policy forum |
| `red_team` | Org A red-teamed Org B's model | Gryphon red-teamed GPT-4 |
| `published_with` | Orgs co-authored a paper/report | OpenAI + RAND bio risk study |
| `advisory` | Org A advises Org B on biosecurity | NIST advises SecureBio on standards |

**Key question for review:** Should these be new edge types or mapped to existing ones?

In [6]:
# === COMPREHENSIVE DRY RUN STATS ===

print('=' * 70)
print('SPRINT 5 DRY RUN — EVIDENCE & STATS REPORT')
print('=' * 70)

# 1. Relationship map
print('\n📊 VERIFIED BIOSECURITY EVALUATION RELATIONSHIPS')
print('-' * 50)
eval_pairs = []
for name, data in FINDINGS.items():
    for r in data['relationships']:
        if 'evaluator' in r:
            eval_pairs.append((r['evaluator'], r['evaluated'], r['confidence']))
            
# Deduplicate and count
from collections import Counter
evaluator_counts = Counter(e[0] for e in eval_pairs)
evaluated_counts = Counter(e[1] for e in eval_pairs)

print(f'\nEvaluator orgs (who does the testing):')
for org, count in evaluator_counts.most_common():
    print(f'  {org}: evaluated {count} AI lab(s)')

print(f'\nEvaluated labs (whose models are tested):')
for lab, count in evaluated_counts.most_common():
    print(f'  {lab}: tested by {count} evaluator(s)')

# 2. Node analysis
print('\n📊 NODE IMPACT ANALYSIS')
print('-' * 50)

# New nodes needed
new_orgs_needed = set()
for name, data in FINDINGS.items():
    for r in data['relationships']:
        for actor in [r.get('evaluator', ''), r.get('evaluated', ''), r.get('convener', '')]:
            if actor and actor != 'multiple LLMs' and actor != 'multiple labs':
                label_lower = actor.lower()
                if label_lower not in node_by_label_lower:
                    # Check partial match
                    found = False
                    for existing_label in node_by_label_lower:
                        if label_lower in existing_label or existing_label in label_lower:
                            found = True
                            break
                    if not found:
                        new_orgs_needed.add(actor)

print(f'New org nodes needed (not in graph): {len(new_orgs_needed)}')
for org in sorted(new_orgs_needed):
    print(f'  → {org}')

# Existing nodes that would gain new edges
existing_nodes_gaining_edges = set()
for name, data in FINDINGS.items():
    for r in data['relationships']:
        for actor in [r.get('evaluator', ''), r.get('evaluated', ''), r.get('convener', '')]:
            if actor:
                for existing_label, existing_id in node_by_label_lower.items():
                    if actor.lower() in existing_label or existing_label in actor.lower():
                        existing_nodes_gaining_edges.add(existing_id)

print(f'\nExisting nodes that would gain new edges: {len(existing_nodes_gaining_edges)}')
for nid in sorted(existing_nodes_gaining_edges):
    n = node_by_id[nid]
    print(f'  [{n["type"]}] {n["label"]}')

# 3. Edge type proposal
print('\n📊 PROPOSED NEW EDGE TYPES')
print('-' * 50)
print('1. biosecurity_eval   — Org A evaluated Org B\'s model for bio risk')
print('   Direction: evaluator → evaluated_lab')
print('   Evidence: SecureBio→Anthropic, SecureBio→OpenAI, Gryphon→OpenAI, etc.')
print('   Confidence: HIGH (public blog posts, system cards)')
print()
print('2. policy_forum       — Org A convened/participated in biosecurity policy forum')
print('   Direction: convener → participant')
print('   Evidence: NTI AIxBio Forum with DeepMind, OpenAI, Anthropic')
print('   Confidence: MEDIUM (mentioned on NTI site but no formal partnership)')
print()
print('3. published_study    — Org A published bio risk research involving Org B\'s models')
print('   Direction: researcher → AI_lab')
print('   Evidence: RAND studied 30+ models, OpenAI+Gryphon GPT-4 study')
print('   Confidence: HIGH (peer-reviewed/published reports)')

# 4. Sprint 4 gap identified
print('\n⚠️  SPRINT 4 GAP IDENTIFIED')
print('-' * 50)
print('Missing program nodes: SAFE GENES, P3, PREEMPT, PREPARE')
print('13 DARPA PI nodes have affiliations but NO connection to their programs')
print('This should be fixed before Sprint 5 merge.')

# 5. Summary
print('\n📊 SUMMARY')
print('-' * 50)
print(f'Total evidence sources analyzed:     6')
print(f'Total relationships found:           {total_rels}')
print(f'HIGH confidence relationships:       {sum(1 for e in eval_pairs if e[2] == "HIGH")}')
print(f'New nodes needed:                    {len(new_orgs_needed)}')
print(f'Existing nodes gaining edges:        {len(existing_nodes_gaining_edges)}')
print(f'New edge types proposed:             3')
print(f'Sprint 4 gaps to fix:                4 missing program nodes + 13 PI connections')

SPRINT 5 DRY RUN — EVIDENCE & STATS REPORT

📊 VERIFIED BIOSECURITY EVALUATION RELATIONSHIPS
--------------------------------------------------

Evaluator orgs (who does the testing):
  SecureBio: evaluated 4 AI lab(s)
  UK AI Safety Institute: evaluated 2 AI lab(s)
  Gryphon Scientific: evaluated 1 AI lab(s)
  RAND Corporation: evaluated 1 AI lab(s)
  FutureHouse: evaluated 1 AI lab(s)

Evaluated labs (whose models are tested):
  OpenAI: tested by 3 evaluator(s)
  Anthropic: tested by 2 evaluator(s)
  Google DeepMind: tested by 1 evaluator(s)
  xAI: tested by 1 evaluator(s)
  multiple LLMs: tested by 1 evaluator(s)
  multiple labs: tested by 1 evaluator(s)

📊 NODE IMPACT ANALYSIS
--------------------------------------------------
New org nodes needed (not in graph): 1
  → xAI

Existing nodes that would gain new edges: 25
  [author] Quentin Gregory Anthony
  [institution] Anthropic
  [institution] DeepMind
  [institution] Google
  [institution] Google DeepMind
  [institution] OpenAI
  [